# Storm Surge Shelter Demand Model — Full E2E Pipeline

**American Red Cross — Storm Surge Shelter Demand Estimation**

This notebook runs the complete pipeline from NHC advisory to census-tract-level shelter demand:

1. **Download** NHC P-Surge raster for the specified storm/advisory
2. **Download** NSI building inventory for affected states
3. **Filter & Map** buildings via DuckDB (bbox clip, dedup, column mapping)
4. **Run FAST** engine to produce building-level damage predictions
5. **Classify** each building into Slight/Moderate/Extensive/Complete damage states
6. **Aggregate** to census tract level and compute BHI (Building Habitability Index)
7. **Join** Census population + CDC SVI data
8. **Estimate** shelter-seeking population (low/high range)

**How to use:**
1. Paste JSON params from Excel Step 6 into Cell 2
2. Run All Cells
3. Copy output from Cell 13 back into Excel Step 7

In [ ]:
# Cell 1: Environment Setup + All Imports
import subprocess, sys, os, io, re, json, time, tempfile, zipfile, warnings
from pathlib import Path

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'pandas', 'numpy', 'duckdb', 'pyarrow', 'rasterio', 'geopandas',
    'requests', 'openpyxl', 'pygris'])

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import requests
from rasterio.io import MemoryFile
from rasterio.warp import transform_bounds
from requests.adapters import HTTPAdapter
from shapely.geometry import box
from urllib3.util.retry import Retry
from urllib import request as urllib_request

WORK_DIR = Path(tempfile.gettempdir()) / 'bhi_pipeline'
WORK_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = 'https://github.com/alexj11324/TSSPIM-Tsunami-and-Storm-Surge-Population-Impact-Model.git'
REPO_DIR = Path('repo')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'{REPO_DIR}/ already exists, skipping clone.')

print(f'Work dir: {WORK_DIR}')
print('Environment ready.')

In [ ]:
# Cell 2: JSON Parameters from Excel Step 6
# =========================================
# PASTE YOUR PARAMS FROM EXCEL BELOW:

params = {
    # Step 1: Required storm inputs
    'storm_id': 'AL022024',
    'storm_name': 'BERYL',
    'advisory': 29,
    'year': 2024,

    # Step 3: Building Usability Thresholds (optional)
    'BLDNG_USABILITY': {
        'Slight':    {'FU': 1.00, 'PU': 0.00, 'NU': 0.00},
        'Moderate':  {'FU': 0.87, 'PU': 0.13, 'NU': 0.00},
        'Extensive': {'FU': 0.25, 'PU': 0.50, 'NU': 0.25},
        'Complete':  {'FU': 0.00, 'PU': 0.02, 'NU': 0.98},
    },

    # Step 4: Utility Loss Severity (optional) — [low, high] ranges
    'UL_SEVERITY': {
        'low':    {'FU': [0.00, 0.05], 'PU': [0.05, 0.10]},
        'medium': {'FU': [0.00, 0.10], 'PU': [0.30, 0.50]},
        'high':   {'FU': [0.10, 0.30], 'PU': [0.60, 0.80]},
    },

    # Step 5: SVI Mapping — % non-habitable seeking shelter [SVI<0.4, 0.4-0.8, >=0.8]
    'SVI_THRESHOLD': [0.000, 0.025, 0.050],
}

# Damage state thresholds (BldgDmgPct % ranges, Hazus standard defaults)
DAMAGE_STATE_THRESHOLDS = {
    'Slight':    (0, 15),
    'Moderate':  (15, 40),
    'Extensive': (40, 60),
    'Complete':  (60, 100),
}

# Tract severity thresholds (Step 2 from Excel)
TRACT_SEVERITY = {
    'high':   {'pct_destroyed': 0.35, 'pct_major_damage': 0.35},
    'medium': {'pct_destroyed': 0.11, 'pct_major_damage': 0.16},
}

# --- Validate params ---
for state, ratios in params['BLDNG_USABILITY'].items():
    total = sum(ratios.values())
    if abs(total - 1.0) > 0.01:
        raise ValueError(f'Usability ratios for {state} sum to {total}, expected 1.0')

for sev, classes in params['UL_SEVERITY'].items():
    for cls, bounds in classes.items():
        if not (0 <= bounds[0] <= bounds[1] <= 1):
            raise ValueError(f'UL_SEVERITY[{sev}][{cls}] = {bounds} out of [0,1] range')

for v in params['SVI_THRESHOLD']:
    if not 0 <= v <= 1:
        raise ValueError(f'SVI_THRESHOLD value {v} out of [0,1]')

print(f'Storm: {params["storm_name"]} ({params["storm_id"]}), Advisory #{params["advisory"]}, Year {params["year"]}')
print(f'Damage state thresholds: {DAMAGE_STATE_THRESHOLDS}')
print('Params validated OK.')

## Stage 1: Download NHC P-Surge Raster

Downloads the storm surge inundation raster directly from NHC for the specified storm and advisory.
Also identifies which US states are within the raster footprint.

In [ ]:
# Cell 3: Download NHC P-Surge Raster

def _normalize_storm_id(storm_id: str, year: int) -> str:
    storm_id = storm_id.upper()
    match = re.match(r'(?P<basin>[A-Z]{2})(?P<number>\d{1,2})(?P<year>\d{2,4})?$', storm_id)
    if not match:
        raise ValueError(f'Invalid storm_id format: {storm_id!r}. Expected e.g. AL022024')
    basin = match.group('basin')
    number = match.group('number').zfill(2)
    provided_year = match.group('year')
    if provided_year:
        normalized_year = provided_year if len(provided_year) == 4 else f'20{provided_year}'
    else:
        normalized_year = str(year)
    return f'{basin}{number}{normalized_year}'


def download_surge_raster(
    storm_id: str, storm_name: str, adv: int, year: int, timeout: int = 60
) -> tuple[str, list[str]]:
    """Download NHC P-Surge raster and identify affected states."""
    norm_id = _normalize_storm_id(storm_id, year)
    name_upper = storm_name.upper()
    adv_padded = f'{int(adv):03d}'

    url = f'https://www.nhc.noaa.gov/gis/inundation/forecasts/{norm_id}_{adv_padded}_tidalmask.zip'
    tif_name = f'{name_upper}_{year}_adv{int(adv)}_e10_ResultMaskRaster.tif'

    # Download with retries
    retry = Retry(total=3, status_forcelist=[429, 500, 502, 503, 504], backoff_factor=0.5)
    session = requests.Session()
    session.mount('https://', HTTPAdapter(max_retries=retry))

    print(f'Downloading {url} ...')
    resp = session.get(url, stream=True, timeout=timeout)
    resp.raise_for_status()

    # Extract TIF from ZIP
    with zipfile.ZipFile(io.BytesIO(resp.content), 'r') as z:
        if tif_name not in z.namelist():
            raise FileNotFoundError(f'{tif_name} not found in archive (available: {z.namelist()})')
        tif_bytes = z.read(tif_name)

    # Save TIF to disk for FAST engine
    raster_path = str(WORK_DIR / tif_name)
    with open(raster_path, 'wb') as f:
        f.write(tif_bytes)

    # Identify affected states using proper context managers
    with MemoryFile(tif_bytes) as mem_file:
        with mem_file.open() as surge_data:
            surge_box = box(*surge_data.bounds)
            surge_crs = surge_data.crs

    surge_gdf = gpd.GeoDataFrame({'geometry': [surge_box]}, crs=surge_crs)

    from pygris import states as get_states
    us_states = get_states(cb=True, cache=True, year=year).to_crs(surge_crs)
    overlapping = gpd.sjoin(us_states, surge_gdf, how='inner', predicate='intersects')
    state_names = overlapping['NAME'].unique().tolist()

    if not state_names:
        raise RuntimeError('No US states overlap with raster footprint. Check storm_id/advisory.')

    print(f'Raster saved: {raster_path}')
    print(f'Affected states: {state_names}')
    return raster_path, state_names


raster_path, affected_states = download_surge_raster(
    params['storm_id'], params['storm_name'], params['advisory'], params['year']
)

## Stage 2: Download NSI Building Inventory

Downloads NSI (National Structure Inventory) from USACE API for each affected state.
The `cbfips` field is critical for census tract GEOID derivation.

In [ ]:
# Cell 4: Download NSI Data

NSI_API_BASE = 'https://nsi.sec.usace.army.mil/nsiapi/structures'

STATE_FIPS = {
    'Alabama': '01', 'Alaska': '02', 'Arizona': '04', 'Arkansas': '05',
    'California': '06', 'Colorado': '08', 'Connecticut': '09', 'Delaware': '10',
    'Florida': '12', 'Georgia': '13', 'Hawaii': '15', 'Idaho': '16',
    'Illinois': '17', 'Indiana': '18', 'Iowa': '19', 'Kansas': '20',
    'Kentucky': '21', 'Louisiana': '22', 'Maine': '23', 'Maryland': '24',
    'Massachusetts': '25', 'Michigan': '26', 'Minnesota': '27', 'Mississippi': '28',
    'Missouri': '29', 'Montana': '30', 'Nebraska': '31', 'Nevada': '32',
    'New Hampshire': '33', 'New Jersey': '34', 'New Mexico': '35', 'New York': '36',
    'North Carolina': '37', 'North Dakota': '38', 'Ohio': '39', 'Oklahoma': '40',
    'Oregon': '41', 'Pennsylvania': '42', 'Rhode Island': '44', 'South Carolina': '45',
    'South Dakota': '46', 'Tennessee': '47', 'Texas': '48', 'Utah': '49',
    'Vermont': '50', 'Virginia': '51', 'Washington': '53', 'West Virginia': '54',
    'Wisconsin': '55', 'Wyoming': '56',
}

NSI_KEEP_COLS = [
    'bid', 'occtype', 'val_struct', 'sqft', 'num_story', 'found_type',
    'found_ht', 'val_cont', 'cbfips', 'pop2pmu65', 'pop2pmo65',
]


def download_nsi_state(state_name: str) -> pd.DataFrame:
    """Download NSI for one state from USACE API, return DataFrame."""
    fips = STATE_FIPS.get(state_name)
    if not fips:
        warnings.warn(f'No FIPS for {state_name!r}, skipping')
        return pd.DataFrame()

    cache_path = WORK_DIR / f'nsi_{state_name.replace(" ", "_").lower()}.parquet'
    if cache_path.exists():
        print(f'  {state_name}: loading cached {cache_path.name}')
        return pd.read_parquet(cache_path)

    url = f'{NSI_API_BASE}?fips={fips}&fmt=fc'
    print(f'  {state_name} (FIPS={fips}): downloading from NSI API...')

    for attempt in range(3):
        try:
            req = urllib_request.Request(url, headers={'Accept': 'application/json'})
            with urllib_request.urlopen(req, timeout=300) as resp:
                data = json.loads(resp.read().decode('utf-8'))
            break
        except (OSError, json.JSONDecodeError) as e:
            print(f'    Attempt {attempt+1} failed: {e}')
            if attempt < 2:
                time.sleep(5 * (attempt + 1))
            else:
                raise

    # Parse GeoJSON features
    rows = []
    for feat in data.get('features', []):
        props = feat.get('properties', {})
        geom = feat.get('geometry', {})
        row = {k: props.get(k) for k in NSI_KEEP_COLS if k in props}
        if geom.get('type') == 'Point':
            coords = geom['coordinates']
            row['longitude'] = coords[0]
            row['latitude'] = coords[1]
        else:
            row['longitude'] = None
            row['latitude'] = None
        rows.append(row)

    df = pd.DataFrame(rows)
    print(f'    Downloaded {len(df):,} buildings')

    df.to_parquet(cache_path, index=False)
    return df


# Download NSI for all affected states
print(f'Downloading NSI for {len(affected_states)} states...')
nsi_dfs = []
failed_states = []
for state in affected_states:
    try:
        df = download_nsi_state(state)
        if not df.empty:
            nsi_dfs.append(df)
    except Exception as e:
        failed_states.append((state, str(e)))
        print(f'  FAILED: {state} - {e}')

if failed_states:
    warnings.warn(f'NSI download failed for {len(failed_states)} states: {[s for s,_ in failed_states]}')

if not nsi_dfs:
    raise RuntimeError('No NSI data downloaded. Cannot proceed.')

nsi_df = pd.concat(nsi_dfs, ignore_index=True)
print(f'\nTotal NSI buildings: {len(nsi_df):,}')

In [ ]:
# Cell 5: Spatial Filter + FAST Input Preparation

def raster_bbox_wgs84(raster_path: str) -> tuple:
    """Get raster bounding box in WGS84 (min_lon, min_lat, max_lon, max_lat)."""
    with rasterio.open(raster_path) as src:
        if src.crs and str(src.crs) != 'EPSG:4326':
            return transform_bounds(src.crs, 'EPSG:4326', *src.bounds)
        return src.bounds


FOUND_TYPE_MAP = {
    'S': 7, 'SLAB': 7, 'C': 5, 'CRAWL': 5, 'CRAWLSPACE': 5,
    'B': 4, 'BASEMENT': 4, 'P': 2, 'PIER': 2, 'PILE': 2, 'PILES': 2,
}

bbox = raster_bbox_wgs84(raster_path)
print(f'Raster bbox (WGS84): lon=[{bbox[0]:.4f}, {bbox[2]:.4f}], lat=[{bbox[1]:.4f}, {bbox[3]:.4f}]')

# Filter NSI to raster bbox
nsi_filtered = nsi_df[
    nsi_df['latitude'].between(bbox[1], bbox[3]) &
    nsi_df['longitude'].between(bbox[0], bbox[2]) &
    nsi_df['latitude'].notna() &
    nsi_df['bid'].notna()
].copy()

# Dedup by bid (keep highest val_struct)
nsi_filtered = (
    nsi_filtered.sort_values('val_struct', ascending=False)
    .drop_duplicates(subset='bid', keep='first')
)

# Map columns for FAST
fast_input = pd.DataFrame({
    'FltyId': nsi_filtered['bid'],
    'Occ': nsi_filtered['occtype'].str.split('-').str[0].str.upper(),
    'Cost': nsi_filtered['val_struct'].fillna(0),
    'Area': nsi_filtered['sqft'].fillna(0),
    'NumStories': nsi_filtered['num_story'].fillna(1),
    'FoundationType': nsi_filtered['found_type'].map(
        lambda ft: FOUND_TYPE_MAP.get(str(ft).upper().strip(), 7) if pd.notna(ft) else 7
    ),
    'FirstFloorHt': nsi_filtered['found_ht'].fillna(1.0),
    'ContentCost': nsi_filtered['val_cont'].fillna(0),
    'Latitude': nsi_filtered['latitude'],
    'Longitude': nsi_filtered['longitude'],
})

fast_csv_path = str(WORK_DIR / 'fast_input.csv')
fast_input.to_csv(fast_csv_path, index=False)
print(f'\nFAST input: {len(fast_input):,} buildings -> {fast_csv_path}')
print(f'  Residential: {fast_input["Occ"].str.startswith("RES").sum():,}')
fast_input.head()

In [ ]:
# Cell 6: Run FAST Engine

mapping = {
    'UserDefinedFltyId': 'FltyId', 'OCC': 'Occ', 'Cost': 'Cost',
    'Area': 'Area', 'NumStories': 'NumStories',
    'FoundationType': 'FoundationType', 'FirstFloorHt': 'FirstFloorHt',
    'ContentCost': 'ContentCost', 'Latitude': 'Latitude', 'Longitude': 'Longitude',
}

mapping_path = str(WORK_DIR / 'fast_mapping.json')
with open(mapping_path, 'w') as f:
    json.dump(mapping, f)

fast_output_dir = str(WORK_DIR / 'fast_output')
os.makedirs(fast_output_dir, exist_ok=True)

fast_script = str(REPO_DIR / 'FAST-main' / 'Python_env' / 'run_fast.py')
cmd = [
    sys.executable, fast_script,
    '--inventory', fast_csv_path,
    '--mapping-json', mapping_path,
    '--flc', 'CoastalA',
    '--rasters', raster_path,
    '--output-dir', fast_output_dir,
    '--project-root', str(REPO_DIR / 'FAST-main'),
]

print('Running FAST engine...')
result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)

if result.returncode != 0:
    print(f'FAST STDERR (last 2000 chars):\n{result.stderr[-2000:]}')
    raise RuntimeError(f'FAST failed with return code {result.returncode}')

print('FAST completed successfully.')

# Find predictions output
pred_files = sorted(Path(fast_output_dir).rglob('*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if not pred_files:
    raise FileNotFoundError(f'FAST produced no output CSV in {fast_output_dir}')

predictions_path = str(pred_files[0])
print(f'Predictions file: {predictions_path}')

In [ ]:
# Cell 7: Load Predictions + Derive Census Tract GEOID

pred_df = pd.read_csv(predictions_path, dtype={'FltyId': str})
print(f'Loaded {len(pred_df):,} predictions')

if pred_df.empty:
    raise RuntimeError('Predictions file is empty. FAST may have failed silently.')

pred_df.columns = pred_df.columns.str.lower()

# Dedup: MAX(bldgdmgpct) per FltyId
pred_df = (
    pred_df.sort_values('bldgdmgpct', ascending=False)
    .drop_duplicates(subset='fltyid', keep='first')
)
print(f'After dedup: {len(pred_df):,} unique buildings')

# JOIN with NSI to get cbfips
nsi_cbfips = nsi_filtered[['bid', 'cbfips']].rename(columns={'bid': 'fltyid'})
nsi_cbfips['fltyid'] = nsi_cbfips['fltyid'].astype(str)
pred_df['fltyid'] = pred_df['fltyid'].astype(str)

pred_df = pred_df.merge(nsi_cbfips, on='fltyid', how='left')
matched = pred_df['cbfips'].notna().sum()
total = len(pred_df)
print(f'Census block FIPS matched: {matched:,} / {total:,} ({matched/max(total,1)*100:.1f}%)')

# Derive tract GEOID (first 11 digits of 15-digit cbfips)
pred_df['cbfips'] = pred_df['cbfips'].astype(str).str.zfill(15)
pred_df['tract_geoid'] = pred_df['cbfips'].str[:11]

# Filter residential only
pred_df = pred_df[pred_df['occ'].str.startswith('RES', na=False)].copy()
print(f'Residential with tract GEOID: {len(pred_df):,}, unique tracts: {pred_df["tract_geoid"].nunique():,}')

In [ ]:
# Cell 8: Classify Damage States + Aggregate to Census Tract

def classify_damage_state(
    dmg_pct: pd.Series, thresholds: dict = None
) -> pd.Series:
    """Classify BldgDmgPct into Hazus damage states."""
    if thresholds is None:
        thresholds = DAMAGE_STATE_THRESHOLDS
    conditions = [
        dmg_pct >= thresholds['Complete'][0],
        dmg_pct >= thresholds['Extensive'][0],
        dmg_pct >= thresholds['Moderate'][0],
        dmg_pct > 0,
    ]
    choices = ['Complete', 'Extensive', 'Moderate', 'Slight']
    return np.select(conditions, choices, default='None')


# Classify each building
pred_df['damage_state'] = classify_damage_state(pred_df['bldgdmgpct'])
state_dist = pred_df['damage_state'].value_counts()
print('Building damage state distribution:')
for ds in ['Slight', 'Moderate', 'Extensive', 'Complete', 'None']:
    n = state_dist.get(ds, 0)
    print(f'  {ds:>10s}: {n:>8,} ({n/len(pred_df)*100:.1f}%)')

# One-hot encode damage states for fast aggregation
for ds in ['Slight', 'Moderate', 'Extensive', 'Complete']:
    pred_df[f'is_{ds}'] = (pred_df['damage_state'] == ds).astype(int)

# Aggregate to census tract
tract_agg = pred_df.groupby('tract_geoid').agg(
    Total_Num_Building=('fltyid', 'count'),
    max_intensity=('bldgdmgpct', 'max'),
    Total_Num_Building_Slight=('is_Slight', 'sum'),
    Total_Num_Building_Moderate=('is_Moderate', 'sum'),
    Total_Num_Building_Extensive=('is_Extensive', 'sum'),
    Total_Num_Building_Complete=('is_Complete', 'sum'),
).reset_index()

tract_agg['max_intensity'] = tract_agg['max_intensity'] / 100.0

# Destruction rates for severity classification
tract_agg['pct_destroyed'] = tract_agg['Total_Num_Building_Complete'] / tract_agg['Total_Num_Building']
tract_agg['pct_major_damage'] = (
    (tract_agg['Total_Num_Building_Extensive'] + tract_agg['Total_Num_Building_Complete'])
    / tract_agg['Total_Num_Building']
)

print(f'\nAggregated to {len(tract_agg):,} census tracts')
tract_agg.head()

In [ ]:
# Cell 9: Classify Tract Severity (risk_level)

def classify_tract_severity(
    pct_destroyed: pd.Series, pct_major: pd.Series, thresholds: dict = None
) -> pd.Series:
    if thresholds is None:
        thresholds = TRACT_SEVERITY
    conditions = [
        (pct_destroyed >= thresholds['high']['pct_destroyed']) |
        (pct_major >= thresholds['high']['pct_major_damage']),
        (pct_destroyed >= thresholds['medium']['pct_destroyed']) |
        (pct_major >= thresholds['medium']['pct_major_damage']),
    ]
    return np.select(conditions, ['high', 'medium'], default='low')


tract_agg['risk_level'] = classify_tract_severity(
    tract_agg['pct_destroyed'], tract_agg['pct_major_damage']
)

risk_dist = tract_agg['risk_level'].value_counts()
print('Tract severity distribution:')
for level in ['low', 'medium', 'high']:
    n = risk_dist.get(level, 0)
    print(f'  {level:>6s}: {n:>6,} tracts ({n/len(tract_agg)*100:.1f}%)')

In [ ]:
# Cell 10: Compute BHI Factor (vectorized)

usability = params['BLDNG_USABILITY']
ul_severity = params['UL_SEVERITY']
total_bldg = tract_agg['Total_Num_Building'].replace(0, np.nan)

bhi_low = pd.Series(0.0, index=tract_agg.index)
bhi_high = pd.Series(0.0, index=tract_agg.index)

for ds in ['Slight', 'Moderate', 'Extensive', 'Complete']:
    frac = tract_agg[f'Total_Num_Building_{ds}'] / total_bldg
    u = usability[ds]

    # Per-tract severity-dependent UL rates
    fu_low = tract_agg['risk_level'].map(lambda r: ul_severity[r]['FU'][0])
    fu_high = tract_agg['risk_level'].map(lambda r: ul_severity[r]['FU'][1])
    pu_low = tract_agg['risk_level'].map(lambda r: ul_severity[r]['PU'][0])
    pu_high = tract_agg['risk_level'].map(lambda r: ul_severity[r]['PU'][1])

    bhi_low += frac * (u['FU'] * fu_low + u['PU'] * pu_low + u['NU'] * 1.0)
    bhi_high += frac * (u['FU'] * fu_high + u['PU'] * pu_high + u['NU'] * 1.0)

tract_agg['BHI_factor_low'] = bhi_low.fillna(0)
tract_agg['BHI_factor_high'] = bhi_high.fillna(0)

print(f'BHI factor range:')
print(f'  Low:  {tract_agg["BHI_factor_low"].min():.6f} - {tract_agg["BHI_factor_low"].max():.6f}')
print(f'  High: {tract_agg["BHI_factor_high"].min():.6f} - {tract_agg["BHI_factor_high"].max():.6f}')

In [ ]:
# Cell 11: Load Census Population + SVI (Tract Level)

CENSUS_API_BASE = 'https://api.census.gov/data/2022/acs/acs5'
CDC_SVI_URL = 'https://svi.cdc.gov/Documents/Data/2022/csv/SVI_2022_US.csv'


def fetch_tract_population(state_fips_list: list[str]) -> pd.DataFrame:
    all_rows = []
    failed = []
    for state_fips in state_fips_list:
        url = f'{CENSUS_API_BASE}?get=B01001_001E&for=tract:*&in=state:{state_fips}'
        try:
            resp = requests.get(url, timeout=60)
            resp.raise_for_status()
            data = resp.json()
            for row in data[1:]:
                pop, st, county, tract = row
                geoid = f'{st}{county}{tract}'
                all_rows.append({'tract_geoid': geoid, 'population': int(float(pop or 0))})
            print(f'  State {state_fips}: {len(data)-1} tracts')
        except Exception as e:
            failed.append(state_fips)
            print(f'  State {state_fips}: FAILED ({e})')

    if failed:
        warnings.warn(f'Census API failed for states: {failed}')
    return pd.DataFrame(all_rows)


def fetch_tract_svi() -> pd.DataFrame:
    svi_path = WORK_DIR / 'svi_2022_tract.csv'
    if not svi_path.exists():
        print('  Downloading CDC SVI 2022 tract-level data...')
        resp = requests.get(CDC_SVI_URL, timeout=300)
        resp.raise_for_status()
        with open(svi_path, 'wb') as f:
            f.write(resp.content)
        print(f'  Saved to {svi_path}')
    else:
        print('  Loading cached SVI data...')

    df = pd.read_csv(svi_path, dtype={'FIPS': str}, usecols=['FIPS', 'RPL_THEMES'])
    df = df.rename(columns={'FIPS': 'tract_geoid', 'RPL_THEMES': 'SVI_Value'})
    df['tract_geoid'] = df['tract_geoid'].str.zfill(11)
    df['SVI_Value'] = pd.to_numeric(df['SVI_Value'], errors='coerce').fillna(0)
    df.loc[df['SVI_Value'] < 0, 'SVI_Value'] = 0  # sentinel -999
    return df


def map_svi_to_shelter_rate(svi_values: pd.Series, thresholds: list) -> pd.Series:
    """Map SVI score to shelter-seeking rate. Conditions are mutually exhaustive for non-NaN."""
    return np.select(
        [svi_values < 0.4, svi_values < 0.8, svi_values >= 0.8],
        thresholds,
        default=0.0,
    )


# Fetch data
state_fips_in_data = sorted(tract_agg['tract_geoid'].str[:2].unique().tolist())
print(f'Fetching Census population for states: {state_fips_in_data}')
census_df = fetch_tract_population(state_fips_in_data)
print(f'Census data: {len(census_df):,} tracts')

print('\nFetching CDC SVI tract-level data...')
svi_df = fetch_tract_svi()
print(f'SVI data: {len(svi_df):,} tracts')

# Join
tract_agg = tract_agg.merge(census_df, on='tract_geoid', how='left')
tract_agg['population'] = tract_agg['population'].fillna(0).astype(int)

tract_agg = tract_agg.merge(svi_df, on='tract_geoid', how='left')
tract_agg['SVI_Value'] = tract_agg['SVI_Value'].fillna(0)
tract_agg['SVI_Value_Mapped'] = map_svi_to_shelter_rate(
    tract_agg['SVI_Value'], params['SVI_THRESHOLD']
)

pop_matched = (tract_agg['population'] > 0).sum()
svi_matched = (tract_agg['SVI_Value'] > 0).sum()
print(f'\nMatched: {pop_matched}/{len(tract_agg)} tracts with Census pop, {svi_matched}/{len(tract_agg)} with SVI')

In [ ]:
# Cell 12: Compute Shelter-Seeking Population + Overview

tract_agg['shelter_seeking_low'] = (
    tract_agg['population'] * tract_agg['BHI_factor_low'] * tract_agg['SVI_Value_Mapped']
)
tract_agg['shelter_seeking_high'] = (
    tract_agg['population'] * tract_agg['BHI_factor_high'] * tract_agg['SVI_Value_Mapped']
)

# Build final output
output_cols = [
    'tract_geoid', 'max_intensity', 'population', 'Total_Num_Building',
    'risk_level', 'BHI_factor_low', 'BHI_factor_high',
    'shelter_seeking_low', 'shelter_seeking_high',
    'Total_Num_Building_Slight', 'Total_Num_Building_Moderate',
    'Total_Num_Building_Extensive', 'Total_Num_Building_Complete',
    'SVI_Value', 'SVI_Value_Mapped',
]
final_output = tract_agg[output_cols].rename(columns={'tract_geoid': 'GEOID'})

# Overview
n_tracts = len(final_output)
total_pop = final_output['population'].sum()
total_shelter_low = final_output['shelter_seeking_low'].sum()
total_shelter_high = final_output['shelter_seeking_high'].sum()
avg_intensity = final_output['max_intensity'].mean()

w = 60
print(f'{"":=<{w}}')
print(f'{"SHELTER DEMAND MODEL \u2014 OVERVIEW":^{w}}')
print(f'{"":=<{w}}')
print(f'  Storm: {params["storm_name"]} ({params["storm_id"]}), Advisory #{params["advisory"]}')
print(f'{"":_<{w}}')
print(f'  Average Storm Surge Intensity:      {avg_intensity:>12.4f}')
print(f'  Number of Tracts Impacted:          {n_tracts:>12,}')
print(f'  Total Population Impacted:          {total_pop:>12,}')
print(f'  Shelter-Seeking Population (Low):   {total_shelter_low:>12.1f}')
print(f'  Shelter-Seeking Population (High):  {total_shelter_high:>12.1f}')
print(f'{"":_<{w}}')

for cat in ['Slight', 'Moderate', 'Extensive', 'Complete']:
    col = f'Total_Num_Building_{cat}'
    total = final_output[col].sum()
    avg = final_output[col].mean()
    pct = total / max(final_output['Total_Num_Building'].sum(), 1) * 100
    print(f'  {cat:>10s}: avg={avg:>8.1f}  total={total:>10,.0f}  ({pct:.1f}%)')

print(f'{"":=<{w}}')

for level in ['low', 'medium', 'high']:
    s = final_output[final_output['risk_level'] == level]
    print(f'  {level.upper()} risk: {len(s)} tracts, shelter={s["shelter_seeking_low"].sum():.1f}-{s["shelter_seeking_high"].sum():.1f}')

final_output.head(10)

In [ ]:
# Cell 13: Export Results

csv_path = str(WORK_DIR / 'shelter_demand_output.csv')
final_output.to_csv(csv_path, index=False)
print(f'CSV exported: {csv_path} ({len(final_output)} rows)')

xlsx_path = str(WORK_DIR / 'shelter_demand_output.xlsx')
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    final_output.to_excel(writer, sheet_name='Output', index=False)
    pd.DataFrame({
        'Parameter': [
            'Storm ID', 'Storm Name', 'Advisory #', 'Year',
            'Total Tracts', 'Total Population',
            'Shelter-Seeking (Low)', 'Shelter-Seeking (High)',
        ],
        'Value': [
            params['storm_id'], params['storm_name'], params['advisory'], params['year'],
            n_tracts, total_pop,
            round(total_shelter_low, 1), round(total_shelter_high, 1),
        ],
    }).to_excel(writer, sheet_name='Parameters', index=False)

print(f'Excel exported: {xlsx_path}')

# JSON for pasting back into Excel Step 7
json_output = final_output.to_json(orient='records', double_precision=6)
print(f'\n--- JSON output (first 500 chars) ---')
print(json_output[:500])

try:
    from google.colab import files
    files.download(csv_path)
    files.download(xlsx_path)
except ImportError:
    print(f'\nFiles saved to: {csv_path}, {xlsx_path}')

---

## Model Card

**Model**: Shelter Demand Estimator v1.0

**Core Formula**:
```
shelter_seeking = population × BHI_factor × SVI_Value_Mapped

BHI_factor = Σ_d fraction_d × [
    USABILITY[d][FU] × UL_SEVERITY[risk][FU][bound] +
    USABILITY[d][PU] × UL_SEVERITY[risk][PU][bound] +
    USABILITY[d][NU] × 1.0
]
```

**Data Sources**: NHC P-Surge, NSI (USACE), FEMA FAST, Census ACS 5-year, CDC SVI 2022

**Known Limitations**:
1. BldgDmgPct → damage state uses hard thresholds (not fragility curves)
2. Tract population from Census ACS may differ from actual nighttime population
3. SVI mapping uses 3 discrete buckets (0-0.4, 0.4-0.8, 0.8-1.0)